# 01 · Getting started with the minus board

This notebook connects to a minus board (v1.1) over USB and walks through the basic commands.
Every step shows **what the library sent** (`>>>`) and **what the board printed back** (`<<<`), then the Python
object the library parsed from it.

You need: the board on USB-C, the firmware from `firmware/*/build/` flashed (see the repo README, *Getting started*),
and Python with `pyserial`, `numpy` and `matplotlib`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "library"))   # the minus package (or: pip install -e python/library)
import warnings
warnings.filterwarnings("ignore", message="Unable to import Axes3D")   # harmless on mixed matplotlib installs
import numpy as np
import matplotlib.pyplot as plt
from minus import Minus, MinusError
import minus
print("library", minus.__version__, "written for firmware", minus.FIRMWARE)

library 1.1.7 written for firmware {'rp': {'name': 'minus-rp2354', 'version': '1.1.7'}, 'pic': {'name': 'minus-pic32', 'version': '1.1.4'}, 'proto': 14}


## 1. Connect

`Minus()` finds the first Raspberry Pi USB serial port (`/dev/ttyACM0` on Linux). Pass the port explicitly if you
have several boards. On connect it reads both firmware versions and **warns** if they differ from the version the
library was written for. `echo=True` would print every exchange; here we print them on purpose with `dev.last.show()`.

In [2]:
dev = Minus()            # or Minus("/dev/ttyACM0")
print(dev.t.port)

/dev/ttyACM0


## 2. How a command goes over the wire

The board's CLI is line-based and has no prompt, so the library can't tell when a reply is finished just by
looking at it. After each command it sends a second line, `#end<k>`. The firmware handles lines in order and answers
that line with `unknown '#end<k>' — try help`. Everything before that answer is the reply to the command.
`exchange()` is the low-level call that every method uses:

In [3]:
ex = dev.exchange("version")
print("sent:    ", repr(ex.command + "\n#end" + str(dev._seq) + "\n"))
print("received:", ex.lines)
print(f"took {ex.seconds * 1000:.1f} ms")

sent:     'version\n#end5\n'
received: ['minus-rp2354 1.1.7 proto 14 (built Nov 22 2026 00:30:26, pico-sdk 2.1.0)', 'board: minus v1.1, flash 2048 kB', 'chip: RP2354A  chip_id=0x30004927 package_sel=0x1 version=A3', 'unique id: 7D4466C63B374C98']
took 0.7 ms


## 3. `version`: RP2354A firmware, board, chip

The parsed `Version` object has the fields; `.raw` keeps the text it came from.

In [4]:
v = dev.version()
dev.last.show()
v

>>> version
<<< minus-rp2354 1.1.7 proto 14 (built Nov 22 2026 00:30:26, pico-sdk 2.1.0)
<<< board: minus v1.1, flash 2048 kB
<<< chip: RP2354A  chip_id=0x30004927 package_sel=0x1 version=A3
<<< unique id: 7D4466C63B374C98
    (4 lines, 1 ms)


Version(name='minus-rp2354', version='1.1.7', proto=14, built='Nov 22 2026 00:30:26', pico_sdk='2.1.0', board='minus v1.1', flash_kb=2048, chip='RP2354A', chip_id=805325095, package_sel=1, chip_rev='A3', unique_id='7D4466C63B374C98')

## 4. The PIC32AK: `raw ID?`

The RP2354A is the only thing on USB. It talks to the PIC32AK (the ADC side) over a UART link.
`raw <text>` forwards one line to the PIC and prints its one-line reply. `pic_ident()` sends `ID?` that way:

In [5]:
p = dev.pic_ident()
dev.last.show()
p

>>> raw ID?
<<< reply: 'PIC32AK1216GC41048 minus-pic32 1.1.4 proto 14'
    (1 lines, 7 ms)


PicIdent(part='PIC32AK1216GC41048', name='minus-pic32', version='1.1.4', proto=14)

In [6]:
dev.check_firmware(strict=True)      # raises MinusError if either firmware is not the one the library expects

{'rp': ('minus-rp2354', '1.1.7'),
 'pic': ('minus-pic32', '1.1.4'),
 'proto': (14, 14)}

## 5. The command list: `help`

In [7]:
print(dev.help())

minus v1.1 RP2354A CLI
  version     firmware version, chip id, unique id
  rgb R G B [pct]  set the RGB LED (0-255 each): LP5562 U5 over I2C (board 3) at pct % brightness (0-100, default 25 = dim) + the WS2812/SK6812 line (devkit, unscaled), e.g. rgb 255 0 0 100
  reboot      reset the RP2354; 'reboot dfu' = reboot into BOOTSEL/UF2 mode
  ping        send PING n over UART0 (GPIO16/17), expect PONG n from the PIC32
  auto|stop   ping every 500 ms / stop
  reset       pulse PIC MCLR low via GPIO24 (needs R53 <= 470R, see README)
  rdy         read RDY (GPIO7 <- PIC RA7)
  trig        2 us pulse on TRIG (GPIO6 -> PIC RA0)
  raw <text>  send <text>\n to the PIC and print whatever comes back within 200 ms
  stat        counters
  lb          RP UART0 internal loopback self-test
  acq [div [samc [in [navg]]]]  capture 4095 ADC samples on the PIC and print them; div = SCCP1 trigger period in 5 ns counts (4 default = 25 Msps: the 32 ns ADC takes every other 20 ns trigger, uniform 40 ns = the 

## 6. Link tests: `ping`, `stat`, `rdy`, `trig`, `lb`

`ping` sends `PING n` to the PIC and expects `PONG n` back. The reply includes the round-trip time and the RDY
handshake line, which the PIC toggles on each PONG.

In [8]:
r = dev.ping()
dev.last.show()
r

>>> ping
<<< ping 4: ok, 1218 us, rdy=1
    (1 lines, 4 ms)


Ping(seq=4, ok=True, round_trip_us=1218, rdy=1, error=None)

In [9]:
pings = [dev.ping() for _ in range(20)]
rtt = np.array([p.round_trip_us for p in pings])
print(f"{sum(p.ok for p in pings)}/20 ok, round trip {rtt.mean():.0f} ± {rtt.std():.0f} us")
s = dev.stat(); dev.last.show(); s

20/20 ok, round trip 1345 ± 75 us
>>> stat
<<< sent=24 ok=24 bad=0 timeout=0
    (1 lines, 1 ms)


Stat(sent=24, ok=24, bad=0, timeout=0)

`trig` pulses the TRIG line (RP → PIC) for 2 µs. The PIC counts the rising edges, and `trig_count()` reads the
count back with `raw TRIG?`. `rdy()` reads the RDY line (PIC → RP). `loopback()` tests the RP's own UART.

In [10]:
before = dev.trig_count()
dev.trig(); dev.last.show()
dev.trig()
after = dev.trig_count(); dev.last.show()
print("edges counted:", after - before)
print("RDY =", dev.rdy(), "| UART loopback ok:", dev.loopback())

>>> trig
<<< TRIG pulsed
    (1 lines, 1 ms)
>>> raw TRIG?
<<< reply: 'TRIG 2'
    (1 lines, 4 ms)
edges counted: 2


RDY = 1 | UART loopback ok: True


## 7. The RGB LED: `rgb R G B [pct]`

On board v1.1 the LED is driven by an LP5562 (I²C address 0x30). `pct` is the brightness in % (0–100), applied by
scaling the PWM; without it the firmware uses **25 %** (dim). `rgb()` returns True when the LP5562 acknowledged.

In [11]:
import time
for col in [(255, 0, 0), (0, 255, 0), (0, 0, 255)]:
    ok = dev.rgb(*col)                 # default brightness: 25 %
    dev.last.show()
    time.sleep(0.5)
for pct in (5, 25, 100):               # same colour, three brightness levels
    dev.rgb(255, 255, 255, pct)
    dev.last.show()
    time.sleep(0.5)
dev.rgb(0, 0, 0)

>>> rgb 255 0 0
<<< rgb 255 0 0 25% (LP5562 0x30: ok)
    (1 lines, 2 ms)


>>> rgb 0 255 0
<<< rgb 0 255 0 25% (LP5562 0x30: ok)
    (1 lines, 2 ms)


>>> rgb 0 0 255
<<< rgb 0 0 255 25% (LP5562 0x30: ok)
    (1 lines, 2 ms)


>>> rgb 255 255 255 5
<<< rgb 255 255 255 5% (LP5562 0x30: ok)
    (1 lines, 2 ms)


>>> rgb 255 255 255 25
<<< rgb 255 255 255 25% (LP5562 0x30: ok)
    (1 lines, 2 ms)


>>> rgb 255 255 255 100
<<< rgb 255 255 255 100% (LP5562 0x30: ok)
    (1 lines, 2 ms)


True

## 8. Errors

When the board answers with a `usage:` line (or `unknown '…'`), the library raises `MinusError` with that text.
The board's state is not changed.

In [12]:
try:
    dev.gain(7)            # legs are 0..3
except MinusError as e:
    print("MinusError:", e)
dev.last.show()

MinusError: 'gain' -> usage: gain [leg 0..3 [u1 0|1 [u2 0|1]]]   leg: Rf 2.2k/4.7k/6.8k/10k (x3.2/x5.7/x7.8/x11 per stage); u = 1: that stage x1
>>> gain 7
<<< usage: gain [leg 0..3 [u1 0|1 [u2 0|1]]]   leg: Rf 2.2k/4.7k/6.8k/10k (x3.2/x5.7/x7.8/x11 per stage); u = 1: that stage x1
    (1 lines, 2 ms)


## 9. History

Every exchange is kept in `dev.history`. That is handy for a lab notebook, or to see exactly what a method sent.

In [13]:
for ex in dev.history[-5:]:
    print(f"{ex.command!r:18} -> {len(ex.lines):5d} lines in {ex.seconds * 1000:6.1f} ms")

'rgb 255 255 255 5' ->     1 lines in    1.8 ms
'rgb 255 255 255 25' ->     1 lines in    1.9 ms
'rgb 255 255 255 100' ->     1 lines in    1.7 ms
'rgb 0 0 0'        ->     1 lines in    1.8 ms
'gain 7'           ->     1 lines in    2.2 ms


In [14]:
dev.close()